# 08 — Entity Resolution & Canonicalization (Milestone M3)

**DSML stage:** modeling. The LLM emits entity strings as written (\"Taiwan Semiconductor Manufacturing
Company Limited\", \"TSMC\", \"Taiwan Semi\") — all must resolve to **one** canonical graph node, or the
graph fragments and multi-hop traversal silently breaks.

Pipeline (knowledge-base-first, per the feasibility studies):
1. **Canonical dictionary** — authoritative aliases for the 14-company universe + key ecosystem entities
   → `artifacts/canonical_entities.json`
2. **Normalization** — casefold, strip legal suffixes (Inc/Corp/Ltd/...)
3. **Fuzzy fallback** — `difflib` ratio ≥ 0.90 against all known aliases
4. **Precision over recall** — unresolved entities are *dropped from relations* and logged to a report
   (reviewing that report is how the dictionary grows)

In [ ]:
import json
import re
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXTRACTIONS = PROJECT_ROOT / "data/processed/extractions/nvda_extractions.jsonl"
assert EXTRACTIONS.exists(), "Run notebook 07 first — no extractions found."
records = [json.loads(line) for line in EXTRACTIONS.open(encoding="utf-8")]
print(f"{len(records)} extraction records loaded")

## 1. Canonical entity dictionary

`entity_id` for SEC filers is their CIK; ecosystem entities that never file get stable negative ids
(matching notebook 06's convention for Samsung).

In [ ]:
CANONICAL = {
    # canonical_name: {"cik": int|None, "ticker": str|None, "aliases": [...]}
    "Nvidia":    {"cik": 1045810, "ticker": "NVDA", "aliases": ["nvidia corporation", "nvidia corp"]},
    "AMD":       {"cik": 2488,    "ticker": "AMD",  "aliases": ["advanced micro devices"]},
    "Intel":     {"cik": 50863,   "ticker": "INTC", "aliases": ["intel corporation", "intel corp"]},
    "Broadcom":  {"cik": 1730168, "ticker": "AVGO", "aliases": ["broadcom inc"]},
    "Qualcomm":  {"cik": 804328,  "ticker": "QCOM", "aliases": ["qualcomm incorporated"]},
    "TSMC":      {"cik": 1046179, "ticker": "TSM",  "aliases": ["taiwan semiconductor manufacturing company", "taiwan semiconductor", "taiwan semi", "tsmc arizona"]},
    "ASML":      {"cik": 937966,  "ticker": "ASML", "aliases": ["asml holding", "asml holding nv"]},
    "Micron":    {"cik": 723125,  "ticker": "MU",   "aliases": ["micron technology"]},
    "Samsung":   {"cik": None,    "ticker": "SSNLF","aliases": ["samsung electronics", "samsung electronics co"]},
    "Apple":     {"cik": 320193,  "ticker": "AAPL", "aliases": ["apple inc"]},
    "Microsoft": {"cik": 789019,  "ticker": "MSFT", "aliases": ["microsoft corporation", "azure", "microsoft azure"]},
    "Amazon":    {"cik": 1018724, "ticker": "AMZN", "aliases": ["amazon com", "aws", "amazon web services"]},
    "Alphabet":  {"cik": 1652044, "ticker": "GOOGL","aliases": ["google", "google cloud", "google llc"]},
    "Meta":      {"cik": 1326801, "ticker": "META", "aliases": ["meta platforms", "facebook"]},
    # --- ecosystem entities (not in the 14 but structurally important; negative synthetic ids) ---
    "SK Hynix":  {"cik": None, "ticker": None, "aliases": ["sk hynix", "hynix"]},
    "Arm":       {"cik": None, "ticker": None, "aliases": ["arm holdings", "arm limited"]},
    "OpenAI":    {"cik": None, "ticker": None, "aliases": ["open ai"]},
    "Anthropic": {"cik": None, "ticker": None, "aliases": []},
    "Marvell":   {"cik": None, "ticker": None, "aliases": ["marvell technology"]},
    "GlobalFoundries": {"cik": None, "ticker": None, "aliases": ["global foundries"]},
    "Foxconn":   {"cik": None, "ticker": None, "aliases": ["hon hai", "hon hai precision industry"]},
    "CoreWeave": {"cik": None, "ticker": None, "aliases": []},
    "xAI":       {"cik": None, "ticker": None, "aliases": []},
    "Oracle":    {"cik": None, "ticker": None, "aliases": ["oracle corporation", "oracle cloud"]},
    "Tesla":     {"cik": None, "ticker": None, "aliases": []},
    "Huawei":    {"cik": None, "ticker": None, "aliases": ["huawei technologies"]},
}

# assign stable synthetic ids to non-filers: Samsung must match notebook 06 (-1); others follow
synthetic = -1
for name, spec in CANONICAL.items():
    if spec["cik"] is None:
        spec["entity_id"], synthetic = synthetic, synthetic - 1
    else:
        spec["entity_id"] = spec["cik"]

canonical_path = PROJECT_ROOT / "artifacts" / "canonical_entities.json"
canonical_path.write_text(json.dumps(CANONICAL, indent=2), encoding="utf-8")
print(f"{len(CANONICAL)} canonical entities → {canonical_path.relative_to(PROJECT_ROOT)}")

## 2. The resolver

In [ ]:
LEGAL_SUFFIXES = re.compile(
    r"\b(incorporated|corporation|corp|inc|ltd|limited|llc|plc|co|company|holdings?|nv|sa|ag|kk)\b\.?", re.I)

def normalize_name(s: str) -> str:
    s = re.sub(r"[^\w\s]", " ", s.lower())
    s = LEGAL_SUFFIXES.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

ALIAS_LOOKUP = {}
for name, spec in CANONICAL.items():
    for alias in [name] + spec["aliases"]:
        ALIAS_LOOKUP[normalize_name(alias)] = name

def resolve(raw: str, fuzzy_threshold: float = 0.90) -> str | None:
    """Raw entity string -> canonical name, or None if unresolvable."""
    norm = normalize_name(raw)
    if not norm:
        return None
    if norm in ALIAS_LOOKUP:
        return ALIAS_LOOKUP[norm]
    best_name, best_score = None, 0.0
    for alias_norm, name in ALIAS_LOOKUP.items():
        score = SequenceMatcher(None, norm, alias_norm).ratio()
        if score > best_score:
            best_name, best_score = name, score
    return best_name if best_score >= fuzzy_threshold else None

# unit checks — the resolver is pure logic, so test it right here
assert resolve("Taiwan Semiconductor Manufacturing Company Limited") == "TSMC"
assert resolve("TSMC") == "TSMC"
assert resolve("Micron Technology, Inc.") == "Micron"
assert resolve("Amazon Web Services") == "Amazon"
assert resolve("SK hynix Inc.") == "SK Hynix"
assert resolve("Advanced Micro Devices, Inc.") == "AMD"
assert resolve("Some Unknown Widget Maker") is None
print("resolver unit checks passed")

## 3. Resolve all extracted relations (drop-and-log for unresolved)

In [ ]:
resolved_records, dropped = [], []
for rec in records:
    out = dict(rec, relations=[])
    for rel in rec["relations"]:
        src, tgt = resolve(rel["source_entity"]), resolve(rel["target_entity"])
        if src and tgt and src != tgt:
            out["relations"].append(dict(rel, source_canonical=src, target_canonical=tgt))
        else:
            dropped.append({"chunk_id": rec["chunk_id"], **rel,
                            "src_resolved": src, "tgt_resolved": tgt})
    resolved_records.append(out)

out_path = PROJECT_ROOT / "data/processed/extractions/nvda_extractions_resolved.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for rec in resolved_records:
        f.write(json.dumps(rec) + "\n")

dropped_df = pd.DataFrame(dropped)
report_path = PROJECT_ROOT / "data/processed/extractions/resolution_report.parquet"
if len(dropped_df):
    dropped_df.to_parquet(report_path, index=False)

n_kept = sum(len(r["relations"]) for r in resolved_records)
print(f"kept {n_kept} relations, dropped {len(dropped)} (unresolved/self-loop) → {report_path.name}")
if len(dropped_df):
    print("\nTop unresolved entity strings (dictionary growth candidates):")
    unresolved = pd.concat([
        dropped_df.loc[dropped_df["src_resolved"].isna(), "source_entity"],
        dropped_df.loc[dropped_df["tgt_resolved"].isna(), "target_entity"],
    ])
    print(unresolved.value_counts().head(10).to_string())

In [ ]:
# --- M3 (resolution) assertion cell ---
assert canonical_path.exists() and out_path.exists()
kept_rels = [r for rec in resolved_records for r in rec["relations"]]
assert len(kept_rels) > 0, "resolution dropped everything — inspect the report"
canon_names = set(CANONICAL)
assert all(r["source_canonical"] in canon_names and r["target_canonical"] in canon_names for r in kept_rels)
drop_rate = len(dropped) / max(1, len(dropped) + len(kept_rels))
print(f"M3 (resolution) OK — {len(kept_rels)} canonical relations, drop rate {drop_rate:.0%} "
      f"({'review report to grow dictionary' if drop_rate > 0.3 else 'healthy'})")